# Notebook 02 — Deep Learning Baseline (Người 2)

**Chủ đề:** Phân tích cảm xúc tiếng Việt bằng **Deep Learning sequence models**.

**Mục tiêu:** train + so sánh 3 kiến trúc trên UIT-VSFC.

| Model | Embedding | Đặc điểm |
|---|---|---|
| LSTM (1 chiều) | trainable / pretrained | baseline RNN |
| BiLSTM | trainable / pretrained | đọc cả 2 chiều |
| GRU | trainable / pretrained | nhẹ hơn LSTM |

**Yêu cầu trước:** chạy `00_data_preparation.ipynb`.

**Đầu ra:**
- Biểu đồ loss/accuracy theo epoch.
- Bảng so sánh 3 model.
- File `models/dl_baseline.pt` + `models/dl_baseline_vocab.json` (best model).

In [ ]:
# --- Imports & paths --------------------------------------------------------
import json, random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix,
)

ROOT = Path('..').resolve()
PROC = ROOT / 'data' / 'processed'
MODELS = ROOT / 'models'; MODELS.mkdir(parents=True, exist_ok=True)
RESULTS = ROOT / 'results'; RESULTS.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
LABEL_NAMES = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
NUM_CLASSES = 3
print('Device:', DEVICE)

In [ ]:
# Load 3 split + xây vocab từ train
def load_split(name):
    df = pd.read_csv(PROC / f'{name}.csv')
    df['text_clean'] = df['text_clean'].fillna(' ').astype(str)
    df['label'] = df['label'].astype(int)
    return df

df_train = load_split('train'); df_dev = load_split('dev'); df_test = load_split('test')

PAD, UNK = '<pad>', '<unk>'
def build_vocab(texts, min_freq=2, max_size=30_000):
    cnt = {}
    for t in texts:
        for w in t.split():
            cnt[w] = cnt.get(w, 0) + 1
    items = sorted([(w, c) for w, c in cnt.items() if c >= min_freq],
                   key=lambda x: -x[1])[:max_size]
    w2i = {PAD: 0, UNK: 1}
    for w, _ in items: w2i[w] = len(w2i)
    return w2i

word2idx = build_vocab(df_train['text_clean'])
VOCAB_SIZE = len(word2idx)
MAX_LEN = max(20, int(df_train['text_clean'].str.split().str.len().quantile(0.95)))
print(f'vocab={VOCAB_SIZE}, max_len={MAX_LEN}')

def encode(t):
    ids = [word2idx.get(w, word2idx[UNK]) for w in t.split()][:MAX_LEN]
    return ids + [0] * (MAX_LEN - len(ids))

class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.X = torch.tensor([encode(t) for t in texts], dtype=torch.long)
        self.y = torch.tensor(list(labels), dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

BATCH = 32
train_dl = DataLoader(TextDataset(df_train['text_clean'], df_train['label']), batch_size=BATCH, shuffle=True)
dev_dl   = DataLoader(TextDataset(df_dev['text_clean'],   df_dev['label']),   batch_size=BATCH)
test_dl  = DataLoader(TextDataset(df_test['text_clean'],  df_test['label']),  batch_size=BATCH)

## Định nghĩa 3 model RNN dùng chung 1 khung

Trừu tượng hoá kiến trúc `Embedding → RNN → MeanPool → Linear` rồi cho `rnn_type` chọn 
LSTM / GRU / BiLSTM. KISS — tránh copy code 3 lần.

In [ ]:
class RNNClassifier(nn.Module):
    """Khung chung cho LSTM/GRU/BiLSTM. Đổi `rnn_type` & `bidir` là ra model khác."""
    def __init__(self, vocab, embed=128, hidden=128, n_classes=NUM_CLASSES,
                 rnn_type='lstm', bidir=False, dropout=0.4):
        super().__init__()
        self.emb = nn.Embedding(vocab, embed, padding_idx=0)
        rnn_cls = {'lstm': nn.LSTM, 'gru': nn.GRU}[rnn_type]
        self.rnn = rnn_cls(embed, hidden, batch_first=True, bidirectional=bidir)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden * (2 if bidir else 1), n_classes)
    def forward(self, x):
        mask = (x != 0).unsqueeze(-1).float()
        h = self.emb(x)
        out, _ = self.rnn(h)
        # Mean-pool có mask để bỏ qua pad
        pooled = (out * mask).sum(1) / mask.sum(1).clamp(min=1)
        return self.fc(self.dropout(pooled))

def train_one(model_factory, name, epochs=10, lr=1e-3, patience=3):
    """Train + early-stopping theo dev acc. Trả về (model, history, metrics_dict)."""
    model = model_factory().to(DEVICE)
    optim_ = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss()
    hist = {'train_loss': [], 'dev_loss': [], 'train_acc': [], 'dev_acc': []}

    def run(loader, train):
        model.train() if train else model.eval()
        tot, cor, n = 0.0, 0, 0
        ctx = torch.enable_grad() if train else torch.no_grad()
        with ctx:
            for X, y in loader:
                X, y = X.to(DEVICE), y.to(DEVICE)
                logits = model(X); loss = crit(logits, y)
                if train:
                    optim_.zero_grad(); loss.backward(); optim_.step()
                tot += loss.item() * X.size(0)
                cor += (logits.argmax(1) == y).sum().item()
                n += X.size(0)
        return tot / n, cor / n

    best, best_state, bad = -1, None, 0
    for ep in range(1, epochs + 1):
        tl, ta = run(train_dl, True); dl, da = run(dev_dl, False)
        hist['train_loss'].append(tl); hist['dev_loss'].append(dl)
        hist['train_acc'].append(ta);  hist['dev_acc'].append(da)
        print(f'  [{name}] ep{ep:02d} loss={tl:.3f}/{dl:.3f} acc={ta:.3f}/{da:.3f}')
        if da > best:
            best, best_state, bad = da, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= patience:
                print(f'  early stop ep{ep}')
                break

    model.load_state_dict(best_state)
    # Evaluate test
    model.eval(); preds, ys = [], []
    with torch.no_grad():
        for X, y in test_dl:
            preds.append(model(X.to(DEVICE)).argmax(1).cpu().numpy())
            ys.append(y.numpy())
    y_pred = np.concatenate(preds); y_true = np.concatenate(ys)
    acc = accuracy_score(y_true, y_pred)
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
    return model, hist, {'model': name, 'accuracy': acc, 'precision': p,
                         'recall': r, 'f1_macro': f1, 'y_pred': y_pred, 'y_true': y_true}

In [ ]:
configs = {
    'LSTM':   lambda: RNNClassifier(VOCAB_SIZE, rnn_type='lstm', bidir=False),
    'BiLSTM': lambda: RNNClassifier(VOCAB_SIZE, rnn_type='lstm', bidir=True),
    'GRU':    lambda: RNNClassifier(VOCAB_SIZE, rnn_type='gru',  bidir=False),
}

all_results = {}
for name, factory in configs.items():
    print(f'\n--- Training {name} ---')
    model, hist, m = train_one(factory, name)
    all_results[name] = (model, hist, m)

In [ ]:
# --- Vẽ loss/acc cho cả 3 model lên cùng figure để dễ so sánh -----------
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for name, (_, hist, _) in all_results.items():
    eps = range(1, len(hist['dev_loss']) + 1)
    axes[0].plot(eps, hist['dev_loss'], label=f'{name} dev')
    axes[1].plot(eps, hist['dev_acc'],  label=f'{name} dev')
axes[0].set_title('Dev Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()
axes[1].set_title('Dev Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].legend()
plt.tight_layout(); plt.savefig(RESULTS / 'dl_curves.png', dpi=120); plt.show()

# Bảng so sánh
dl_results = pd.DataFrame([
    {k: v for k, v in m.items() if k not in ('y_true', 'y_pred')}
    for _, _, m in all_results.values()
]).sort_values('f1_macro', ascending=False).reset_index(drop=True)
print(dl_results)
dl_results.to_csv(RESULTS / 'dl_baseline.csv', index=False)

In [ ]:
# Lưu best model + vocab cho dashboard / báo cáo
best_name = dl_results.iloc[0]['model']
best_model, _, _ = all_results[best_name]
torch.save(best_model.state_dict(), MODELS / 'dl_baseline.pt')
with open(MODELS / 'dl_baseline_vocab.json', 'w', encoding='utf-8') as f:
    json.dump({
        'word2idx': word2idx, 'max_len': MAX_LEN, 'best_model': best_name,
        'embed_dim': 128, 'hidden': 128, 'dropout': 0.4, 'num_classes': NUM_CLASSES,
    }, f, ensure_ascii=False)
print(f'[saved] best={best_name}')

## TODO của Người 2

- [ ] Thử Word2Vec / FastText pretrained (PhoW2V) và nạp vào `self.emb` (freeze 5 epoch đầu, sau đó fine-tune).
- [ ] Tune `embed_dim`, `hidden`, `dropout`, `lr`.
- [ ] Vẽ confusion matrix cho best model.
- [ ] Thêm vào báo cáo: **điểm mạnh** (hiểu ngữ cảnh, capture sequence) / **điểm yếu** (cần nhiều data, chậm).